# MEN 4338 · Week 6 — Classical estimators and cross-validation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/iugb/men4338/notebooks/MEN4338_W6_idw_cross_validation.ipynb)

**IUGB · BSc Mining Engineering · Geostatistics (lecture & lab)** · Instructor: Laurent Kouadio

Before kriging, engineers estimated grades with simple geometric rules: take the **nearest** sample, or
an **inverse-distance** weighted average. They are still used, as quick checks and in many mine software
defaults. This week we program them, and learn how to judge any estimator with **cross-validation**: hide
one sample, estimate it from the others, compare.

For Bassam we can go further: we know the true grades (`bassam_truth_grid.csv`, one value per 10 m cell),
which a real mine never does.

**What you will practise:** nearest neighbour · inverse distance weighting (IDW) · search neighbourhood ·
leave-one-out cross-validation · bias, mean squared error, correlation.

Run each cell with **Shift + Enter**, from top to bottom, starting with the two setup cells.

In [ ]:
# Setup: run this cell first (Colab or local Jupyter).
# It fetches the data and the answer checks: from the repository (../data, ../tests)
# if you downloaded it, otherwise from GitHub.
import os, shutil, urllib.request
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/iugb/men4338/'
FILES = [('data/bassam_samples.csv', 'bassam_samples.csv'), ('data/bassam_truth_grid.csv', 'bassam_truth_grid.csv'), ('tests/w6/q1.py', 'tests/q1.py'), ('tests/w6/q2.py', 'tests/q2.py'), ('tests/w6/q3.py', 'tests/q3.py'), ('tests/w6/q4.py', 'tests/q4.py'), ('tests/w6/q5.py', 'tests/q5.py')]
os.makedirs('tests', exist_ok=True)
for source, target in FILES:
    if os.path.exists(target):
        continue
    local = os.path.join('..', source)
    if os.path.exists(local):
        shutil.copy(local, target)                      # offline kit / cloned repo
    else:
        urllib.request.urlretrieve(URL + source, target)  # Colab: download
print('Ready:', ', '.join(t for _, t in FILES))

In [ ]:
# Automatic answer checks (Otter-Grader)
try:
    import otter
except ImportError:
    %pip install -q otter-grader
    import otter
grader = otter.Notebook(tests_dir="tests")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

samples = pd.read_csv("bassam_samples.csv")
truth = pd.read_csv("bassam_truth_grid.csv")
x, y, au = samples["x"].to_numpy(), samples["y"].to_numpy(), samples["au_gpt"].to_numpy()
T = truth["au_gpt"].to_numpy().reshape(100, 120)      # rows = north, columns = east
print(len(samples), "samples;", T.shape, "truth cells; true mean", T.mean().round(3))

## 1. Every estimator is a weighted average

$$z^*(\mathbf{x}_0) = \sum_{i=1}^{n} \lambda_i \, z(\mathbf{x}_i), \qquad \sum_i \lambda_i = 1$$

* **Nearest neighbour (polygons):** weight 1 for the closest sample, 0 for the others.
* **Inverse distance:** $\lambda_i \propto 1/d_i^{\,p}$, usually $p = 2$, over the $n_{max}$ closest samples.

Neither uses the variogram: they ignore the nugget, the anisotropy and the redundancy between clustered
samples. Kriging (week 8) fixes all three.

## 2. Cross-validation

For each sample $i$: remove it, estimate it from the others, record the error $e_i = z^*_i - z_i$. Then:

| statistic | ideal | tells you |
|---|---|---|
| mean error $\bar e$ | 0 | global bias |
| mean squared error $\overline{e^2}$ | small | precision |
| correlation $\rho(z^*, z)$ | close to 1 | how well the estimates follow reality |

Cross-validation compares estimators on the **same** data. It is not a proof that the model is right.

---
## Exercises

### Q1. Nearest neighbour

Write `nearest_estimate(x0, y0, x, y, v)` returning the value of the closest sample. Store the estimate at
(500, 500) in `nn_500`.

In [ ]:
def nearest_estimate(x0, y0, x, y, v):
    ...

nn_500 = ...
nn_500

In [ ]:
grader.check("q1")

### Q2. Inverse distance

Write `idw_estimate(x0, y0, x, y, v, power=2, nmax=12)` using the `nmax` closest samples. If a sample sits
exactly at the target (distance 0), return its value.

In [ ]:
def idw_estimate(x0, y0, x, y, v, power=2, nmax=12):
    ...

idw_estimate(500, 500, x, y, au)

In [ ]:
grader.check("q2")

### Q3. Compare with the truth

`targets = [(312.0, 457.0), (641.0, 213.0), (906.0, 782.0), (152.0, 933.0), (1083.0, 97.0)]`. Store the IDW estimates (power 2, 12 samples) in `idw_at_targets`, the true grades in
`true_at_targets` (the value of the closest truth-grid cell centre), and the mean absolute error in `mae_targets`.

In [ ]:
targets = [(312.0, 457.0), (641.0, 213.0), (906.0, 782.0), (152.0, 933.0), (1083.0, 97.0)]
idw_at_targets = ...
true_at_targets = ...
mae_targets = ...
list(zip(idw_at_targets, true_at_targets)), mae_targets

In [ ]:
grader.check("q3")

### Q4. Leave-one-out cross-validation

Cross-validate IDW (power 2, 12 samples) on all samples. Store the errors (estimate − true) in `cv_err`
(same order as `samples`), the mean error in `cv_me`, the mean squared error in `cv_mse` and the correlation
between estimates and true values in `cv_rho`. Plot estimates against true values.

In [ ]:
cv_err = ...
cv_me = ...
cv_mse = ...
cv_rho = ...
cv_me, cv_mse, cv_rho

In [ ]:
grader.check("q4")

### Q5. Which power?

Repeat the cross-validation for powers 1, 2 and 3. Store the MSE of each in the dictionary
`cv_mse_by_power` and the best power in `best_power`. What happens to the estimates when the power is very
large?

In [ ]:
cv_mse_by_power = {}
...
best_power = ...
cv_mse_by_power, best_power

In [ ]:
grader.check("q5")

### Going further (not checked)

* Estimate every 10 m cell of the bench by IDW and draw the map next to the truth (`plt.imshow(T, origin="lower")`).
  Which one is smoother? Compare the variance of the estimates with the variance of the truth.
* Do the cross-validation errors depend on the campaign (grid or infill)?

---
**Before you leave:** run all the checks with the cell below, then save your notebook
(**File → Download → .ipynb** in Colab).

In [ ]:
grader.check_all()